In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"

# Customer Churn Prediction & Retention Intelligence

## Project Overview
Build a predictive analytics solution to identify customers likely to discontinue service, understand churn drivers, and recommend data-driven retention actions.

**Dataset**: Telco Customer Churn (7,043 customers, 20 features)

**Target**: Binary classification (Churn: Yes/No)

In [ ]:
# Load data
df = pd.read_csv("/Users/premkumarreddy/Downloads/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
# Drop customerID (not predictive)
df.drop(columns=['customerID'], inplace=True)
df.head()

## 1. Data Quality & Preprocessing

In [ ]:
# Check unique values for categorical columns
for col in df.columns:
    if col not in ['tenure', 'MonthlyCharges', 'TotalCharges']:
        print(f"{col}: {df[col].unique()}")
        print("-------------------------------------")

In [ ]:
# Check missing values
print("Missing values:\n", df.isnull().sum())

# Check for whitespace in numerical columns
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    space_count = len(df[df[col] == " "])
    print(f"{col} whitespace count: {space_count}")

In [ ]:
# Fix TotalCharges: replace whitespace with 0 and convert to float
df['TotalCharges'] = df['TotalCharges'].replace({" ": "0.0"}).astype(float)

df.info()

In [ ]:
# Target distribution
print(df['Churn'].value_counts())
print(f"Churn rate: {df['Churn'].value_counts(normalize=True)['Yes']*100:.1f}%")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary
df.describe().T

In [ ]:
def plot_distribution(df, column_name):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[column_name], kde=True)
    plt.title(f"Distribution of {column_name}")
    
    col_mean = df[column_name].mean()
    col_median = df[column_name].median()
    plt.axvline(col_mean, color="red", linestyle="--", label="Mean")
    plt.axvline(col_median, color="blue", linestyle="--", label="Median")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    sns.boxplot(y=df[column_name])
    plt.title(f"Boxplot of {column_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_distribution(df, "tenure")

In [ ]:
plot_distribution(df, "MonthlyCharges")

In [ ]:
plot_distribution(df, "TotalCharges")

In [ ]:
# Correlation matrix for numerical features
plt.figure(figsize=(8, 6))
sns.heatmap(df[['tenure', 'MonthlyCharges', 'TotalCharges']].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix - Numerical Features")
plt.show()

In [ ]:
# Categorical distributions by Churn
categorical_cols = df.select_dtypes(include="object").columns.tolist() + ['SeniorCitizen']

fig, axes = plt.subplots(5, 3, figsize=(18, 24))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    if idx < len(axes):
        sns.countplot(data=df, x=col, hue='Churn', ax=axes[idx])
        axes[idx].set_title(f"{col} by Churn")
        axes[idx].tick_params(axis='x', rotation=45)

# Hide unused subplots
for idx in range(len(categorical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Statistical Analysis & Feature Engineering

In [ ]:
# IQR Outlier Detection for numerical features
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return outliers, lower_bound, upper_bound

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df[col])
    print(f"{col}: {len(outliers)} outliers detected (bounds: {lower:.2f}, {upper:.2f})")
    print(f"  Outlier %: {len(outliers)/len(df)*100:.2f}%")

In [ ]:
# Feature Engineering
df_fe = df.copy()

# 1. Tenure bands
df_fe['tenure_band'] = pd.cut(df_fe['tenure'], 
                               bins=[0, 12, 24, 48, 72], 
                               labels=['0-1yr', '1-2yr', '2-4yr', '4+yr'],
                               right=True)

# 2. Average monthly charges (customer value proxy)
df_fe['avg_monthly_charges'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)

# 3. High value customer flag (top 25% by TotalCharges)
df_fe['high_value_customer'] = (df_fe['TotalCharges'] > df_fe['TotalCharges'].quantile(0.75)).astype(int)

# 4. Service count (number of active services)
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

def count_active_services(row):
    count = 0
    for col in service_cols:
        val = row[col]
        if val not in ['No', 'No internet service', 'No phone service']:
            count += 1
    return count

df_fe['service_count'] = df_fe.apply(count_active_services, axis=1)

# 5. Contract type encoding (ordinal: month-to-month < one year < two year)
contract_map = {'Month-to-month': 1, 'One year': 2, 'Two year': 3}
df_fe['contract_encoded'] = df_fe['Contract'].map(contract_map)

# 6. Payment method risk (electronic check higher risk)
payment_risk = {'Electronic check': 3, 'Mailed check': 2, 'Bank transfer (automatic)': 1, 'Credit card (automatic)': 1}
df_fe['payment_risk_score'] = df_fe['PaymentMethod'].map(payment_risk)

# 7. Senior citizen with dependents interaction
df_fe['senior_with_dependents'] = ((df_fe['SeniorCitizen'] == 1) & (df_fe['Dependents'] == 'Yes')).astype(int)

# 8. Paperless billing + electronic check interaction (high risk combo)
df_fe['paperless_electronic'] = ((df_fe['PaperlessBilling'] == 'Yes') & (df_fe['PaymentMethod'] == 'Electronic check')).astype(int)

print(f"Original features: {df.shape[1]}")
print(f"Engineered features: {df_fe.shape[1]}")
print(f"New features: {list(set(df_fe.columns) - set(df.columns))}")

df_fe.head()

In [ ]:
# Churn rate by new features
new_cat_features = ['tenure_band', 'high_value_customer', 'service_count', 'contract_encoded', 'payment_risk_score', 'senior_with_dependents', 'paperless_electronic']

for col in new_cat_features:
    if col in df_fe.columns:
        print(f"\n{col} vs Churn:")
        print(df_fe.groupby(col)['Churn'].apply(lambda x: (x=='Yes').mean()).round(3))

## 4. Data Preparation for Modeling

In [ ]:
# Encode target
df_fe['Churn'] = df_fe['Churn'].replace({'Yes': 1, 'No': 0})

# Separate features and target
X = df_fe.drop(columns=['Churn'])
y = df_fe['Churn']

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")

In [ ]:
# Label encode categorical features
from sklearn.preprocessing import LabelEncoder

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# Save encoders
import joblib
joblib.dump(encoders, "encoders.joblib")

X.head()

In [ ]:
# Scale numerical features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Save scaler
joblib.dump(scaler, "scaler.joblib")

X.head()

In [ ]:
# Train-test split with stratification
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate: {y_test.mean():.3f}")

In [ ]:
# Handle class imbalance with SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original train: {y_train.value_counts().to_dict()}")
print(f"SMOTE train: {y_train_smote.value_counts().to_dict()}")

## 5. Predictive Modeling - Multiple Classifiers with Stratified CV

In [ ]:
# Import all required models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, precision_recall_curve, f1_score,
                             precision_score, recall_score, make_scorer)
import time

In [ ]:
# Define base models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    "Neural Network": MLPClassifier(max_iter=500, early_stopping=True, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
}

# Hyperparameter grids
hyperparameters = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ['l2'],
        "solver": ['lbfgs', 'liblinear']
    },
    "Decision Tree": {
        "max_depth": [3, 5, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "criterion": ['gini', 'entropy']
    },
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    },
    "Neural Network": {
        "hidden_layer_sizes": [(50,), (100,), (50, 25), (100, 50)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01]
    },
    "XGBoost": {
        "learning_rate": [0.01, 0.1],
        "max_depth": [3, 5, 7],
        "n_estimators": [100, 200],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "scale_pos_weight": [1, 2]
    }
}

In [ ]:
# Stratified K-Fold for cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Custom scorer for recall (important for churn detection)
recall_scorer = make_scorer(recall_score)

best_models = {}
cv_results = {}

for model_name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Tuning {model_name}...")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    grid = GridSearchCV(
        estimator=model,
        param_grid=hyperparameters[model_name],
        cv=skf,
        scoring='recall',  # Optimize for recall (catch churners)
        n_jobs=-1,
        verbose=0
    )
    
    grid.fit(X_train_smote, y_train_smote)
    
    elapsed = time.time() - start_time
    
    best_models[model_name] = grid.best_estimator_
    cv_results[model_name] = {
        'best_params': grid.best_params_,
        'best_recall': grid.best_score_,
        'best_estimator': grid.best_estimator_,
        'cv_results': grid.cv_results_,
        'train_time': elapsed
    }
    
    print(f"Best parameters: {grid.best_params_}")
    print(f"Best CV Recall: {grid.best_score_:.4f}")
    print(f"Training time: {elapsed:.1f}s")

In [ ]:
# Cross-validation comparison on original (non-SMOTE) training data for fair comparison
print("\n" + "="*60)
print("Stratified CV Comparison (on original training data)")
print("="*60)

cv_comparison = {}

for model_name, best_model in best_models.items():
    # Evaluate with multiple metrics
    cv_recall = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='recall', n_jobs=-1)
    cv_precision = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='precision', n_jobs=-1)
    cv_f1 = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    cv_roc_auc = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
    cv_accuracy = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)
    
    cv_comparison[model_name] = {
        'recall_mean': cv_recall.mean(),
        'recall_std': cv_recall.std(),
        'precision_mean': cv_precision.mean(),
        'precision_std': cv_precision.std(),
        'f1_mean': cv_f1.mean(),
        'f1_std': cv_f1.std(),
        'roc_auc_mean': cv_roc_auc.mean(),
        'roc_auc_std': cv_roc_auc.std(),
        'accuracy_mean': cv_accuracy.mean(),
        'accuracy_std': cv_accuracy.std()
    }
    
    print(f"\n{model_name}:")
    print(f"  Accuracy:  {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
    print(f"  Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})")
    print(f"  Recall:    {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})")
    print(f"  F1-Score:  {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
    print(f"  ROC-AUC:   {cv_roc_auc.mean():.4f} (+/- {cv_roc_auc.std():.4f})")

In [ ]:
# Visualize CV results
import pandas as pd

cv_df = pd.DataFrame(cv_comparison).T
cv_df = cv_df[['accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'roc_auc_mean']]
cv_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

plt.figure(figsize=(12, 6))
cv_df.plot(kind='bar', figsize=(12, 6))
plt.title("Model Comparison - Stratified 5-Fold CV Metrics")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(cv_df.round(4))

## 6. Threshold Tuning for Retention Team Capacity

In [ ]:
# Get probabilities for all models on test set
test_probs = {}
test_preds = {}

for model_name, model in best_models.items():
    prob = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    test_probs[model_name] = prob
    test_preds[model_name] = pred

In [ ]:
# Threshold tuning analysis
def analyze_thresholds(y_true, y_prob, model_name, capacity_pct=0.2):
    """Analyze precision/recall at different thresholds.
    capacity_pct: percentage of customers retention team can handle"""
    
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    
    # Add threshold 1.0 for completeness
    prec = np.append(prec, 1.0)
    rec = np.append(rec, 0.0)
    thresh = np.append(thresh, 1.0)
    
    # Calculate number of predicted positives at each threshold
    n_customers = len(y_true)
    capacity_n = int(n_customers * capacity_pct)
    
    # Find threshold that gives ~capacity_n predicted positives
    pred_counts = [(y_prob >= t).sum() for t in thresh]
    
    # Find best threshold near capacity
    idx_capacity = np.argmin(np.abs(np.array(pred_counts) - capacity_n))
    
    # Also find best F1 threshold
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-10)
    idx_f1 = np.argmax(f1_scores)
    
    return {
        'thresholds': thresh,
        'precision': prec,
        'recall': rec,
        'f1': f1_scores,
        'pred_counts': pred_counts,
        'capacity_threshold': thresh[idx_capacity],
        'capacity_precision': prec[idx_capacity],
        'capacity_recall': rec[idx_capacity],
        'capacity_pred_count': pred_counts[idx_capacity],
        'f1_threshold': thresh[idx_f1],
        'f1_precision': prec[idx_f1],
        'f1_recall': rec[idx_f1],
        'f1_score': f1_scores[idx_f1],
        'capacity_n': capacity_n
    }

# Analyze for each model (assuming team can handle 20% of customer base)
capacity_pct = 0.20  # 20% of test set

threshold_analysis = {}
for model_name in best_models.keys():
    threshold_analysis[model_name] = analyze_thresholds(y_test, test_probs[model_name], model_name, capacity_pct)

In [ ]:
# Display threshold recommendations
print(f"Retention team capacity: {capacity_pct*100:.0f}% of customers ({threshold_analysis[list(best_models.keys())[0]]['capacity_n']} customers)")
print("="*80)

for model_name, analysis in threshold_analysis.items():
    print(f"\n{model_name}:")
    print(f"  Default (0.5) threshold:")
    default_pred = (test_probs[model_name] >= 0.5).sum()
    default_prec = precision_score(y_test, test_preds[model_name])
    default_rec = recall_score(y_test, test_preds[model_name])
    print(f"    Predicted churners: {default_pred}, Precision: {default_prec:.3f}, Recall: {default_rec:.3f}")
    
    print(f"  Capacity-aligned threshold ({analysis['capacity_threshold']:.3f}):")
    print(f"    Predicted churners: {analysis['capacity_pred_count']}, Precision: {analysis['capacity_precision']:.3f}, Recall: {analysis['capacity_recall']:.3f}")
    
    print(f"  Best F1 threshold ({analysis['f1_threshold']:.3f}):")
    print(f"    Precision: {analysis['f1_precision']:.3f}, Recall: {analysis['f1_recall']:.3f}, F1: {analysis['f1_score']:.3f}")

In [ ]:
# Plot Precision-Recall curves with capacity threshold
plt.figure(figsize=(14, 10))

for idx, (model_name, analysis) in enumerate(threshold_analysis.items()):
    plt.subplot(2, 3, idx + 1)
    plt.plot(analysis['recall'], analysis['precision'], 'b-', label='PR Curve')
    
    # Mark default threshold
    default_prec = precision_score(y_test, test_preds[model_name])
    default_rec = recall_score(y_test, test_preds[model_name])
    plt.plot(default_rec, default_prec, 'ro', markersize=8, label='Default (0.5)')
    
    # Mark capacity threshold
    plt.plot(analysis['capacity_recall'], analysis['capacity_precision'], 'g^', markersize=10, label=f'Capacity ({analysis["capacity_threshold"]:.2f})')
    
    # Mark best F1 threshold
    plt.plot(analysis['f1_recall'], analysis['f1_precision'], 'ms', markersize=10, label=f'Best F1 ({analysis["f1_threshold"]:.2f})')
    
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} - Precision-Recall Curve')
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 1.05)
    plt.ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
from sklearn.metrics import roc_curve

plt.figure(figsize=(10, 8))

for model_name in best_models.keys():
    fpr, tpr, _ = roc_curve(y_test, test_probs[model_name])
    auc = roc_auc_score(y_test, test_probs[model_name])
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - All Models')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Model Evaluation & Interpretation

In [ ]:
# Detailed evaluation for each model at capacity threshold
print("="*80)
print("DETAILED MODEL EVALUATION AT CAPACITY THRESHOLD")
print("="*80)

evaluation_results = {}

for model_name in best_models.keys():
    analysis = threshold_analysis[model_name]
    threshold = analysis['capacity_threshold']
    
    # Predict at capacity threshold
    y_pred_capacity = (test_probs[model_name] >= threshold).astype(int)
    
    acc = accuracy_score(y_test, y_pred_capacity)
    prec = precision_score(y_test, y_pred_capacity)
    rec = recall_score(y_test, y_pred_capacity)
    f1 = f1_score(y_test, y_pred_capacity)
    roc_auc = roc_auc_score(y_test, test_probs[model_name])
    cm = confusion_matrix(y_test, y_pred_capacity)
    
    evaluation_results[model_name] = {
        'threshold': threshold,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc_auc,
        'confusion_matrix': cm,
        'predicted_churners': y_pred_capacity.sum()
    }
    
    print(f"\n{model_name} (threshold={threshold:.3f}):")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")
    print(f"  Predicted Churners: {y_pred_capacity.sum()} (capacity: {analysis['capacity_n']})")
    print(f"  Confusion Matrix:\n{cm}")
    print(f"  Classification Report:\n{classification_report(y_test, y_pred_capacity)}")

In [ ]:
# Feature Importance Analysis
def get_feature_importance(model, feature_names, model_name):
    """Extract feature importance from various model types."""
    if hasattr(model, 'feature_importances_'):
        # Tree-based models
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        # Linear models
        importance = np.abs(model.coef_[0])
    else:
        # Neural network - use permutation importance
        from sklearn.inspection import permutation_importance
        result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
        importance = result.importances_mean
    
    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
    fi_df = fi_df.sort_values('importance', ascending=False)
    return fi_df

feature_names = X.columns.tolist()

# Get feature importance for tree-based models
for model_name in ['Random Forest', 'XGBoost', 'Decision Tree']:
    if model_name in best_models:
        fi = get_feature_importance(best_models[model_name], feature_names, model_name)
        print(f"\n{model_name} - Top 15 Features:")
        print(fi.head(15).to_string(index=False))
        
        # Plot
        plt.figure(figsize=(10, 6))
        top15 = fi.head(15).iloc[::-1]
        plt.barh(range(len(top15)), top15['importance'])
        plt.yticks(range(len(top15)), top15['feature'])
        plt.xlabel('Importance')
        plt.title(f'{model_name} - Top 15 Feature Importance')
        plt.tight_layout()
        plt.show()

In [ ]:
# Permutation Importance for all models (model-agnostic)
from sklearn.inspection import permutation_importance

print("Computing permutation importance for all models...")
perm_importance = {}

for model_name, model in best_models.items():
    print(f"  {model_name}...")
    result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
    perm_importance[model_name] = pd.DataFrame({
        'feature': feature_names,
        'importance_mean': result.importances_mean,
        'importance_std': result.importances_std
    }).sort_values('importance_mean', ascending=False)

# Display consensus top features
print("\n" + "="*60)
print("CONSENSUS TOP CHURN DRIVERS (Permutation Importance)")
print("="*60)

# Average rank across models
rank_df = pd.DataFrame()
for model_name, df_imp in perm_importance.items():
    df_imp = df_imp.reset_index(drop=True)
    df_imp['rank'] = df_imp.index + 1
    rank_df[model_name] = df_imp.set_index('feature')['rank']

rank_df['avg_rank'] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values('avg_rank')

print("\nTop 15 Consensus Features:")
print(rank_df.head(15).round(2).to_string())

# Plot consensus
top15_consensus = rank_df.head(15).iloc[::-1]
plt.figure(figsize=(10, 8))
plt.barh(range(len(top15_consensus)), top15_consensus['avg_rank'])
plt.yticks(range(len(top15_consensus)), top15_consensus.index)
plt.xlabel('Average Rank (lower = more important)')
plt.title('Consensus Top 15 Churn Drivers (Permutation Importance)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Analysis for best model (XGBoost typically)
try:
    import shap
    
    # Use best model (typically XGBoost)
    best_model_name = max(cv_comparison.keys(), key=lambda k: cv_comparison[k]['roc_auc_mean'])
    print(f"Running SHAP for {best_model_name}...")
    
    # Sample for SHAP (computational efficiency)
    sample_size = min(500, len(X_test))
    X_test_sample = X_test.sample(sample_size, random_state=42)
    
    explainer = shap.TreeExplainer(best_models[best_model_name])
    shap_values = explainer.shap_values(X_test_sample)
    
    # Summary plot
    shap.summary_plot(shap_values, X_test_sample, plot_type="bar", show=False)
    plt.title(f"SHAP Feature Importance - {best_model_name}")
    plt.tight_layout()
    plt.show()
    
    shap.summary_plot(shap_values, X_test_sample, show=False)
    plt.title(f"SHAP Summary - {best_model_name}")
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("SHAP not installed. Install with: pip install shap")

## 8. Churn Driver Analysis & Retention Recommendations

In [ ]:
# Analyze top churn drivers with business context
top_drivers = rank_df.head(10).index.tolist()
print(f"Top 10 Churn Drivers: {top_drivers}")

# For each driver, show churn rate by feature value
original_df = df_fe.copy()  # Before encoding

for driver in top_drivers:
    if driver in original_df.columns:
        print(f"\n{'='*60}")
        print(f"CHURN DRIVER: {driver}")
        print(f"{'='*60}")
        
        if original_df[driver].dtype in ['object', 'category'] or original_df[driver].nunique() < 10:
            churn_by_val = original_df.groupby(driver)['Churn'].agg(['mean', 'count']).round(3)
            churn_by_val.columns = ['Churn_Rate', 'Count']
            print(churn_by_val.to_string())
        else:
            # Numerical - show by quartiles
            original_df[f'{driver}_quartile'] = pd.qcut(original_df[driver], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
            churn_by_quartile = original_df.groupby(f'{driver}_quartile')['Churn'].agg(['mean', 'count']).round(3)
            churn_by_quartile.columns = ['Churn_Rate', 'Count']
            print(churn_by_quartile.to_string())

In [ ]:
# Retention Action Recommendations
print("="*80)
print("DATA-DRIVEN RETENTION ACTION RECOMMENDATIONS")
print("="*80)

recommendations = {
    "Contract Type (Month-to-month)": {
        "insight": "Month-to-month contracts have significantly higher churn (~42%) vs 1-year (~11%) and 2-year (~3%)",
        "actions": [
            "Offer contract upgrade incentives (discounts, free months) for annual/2-year commitments",
            "Create loyalty program with tenure-based rewards",
            "Proactive outreach at 6-month mark for month-to-month customers"
        ]
    },
    "Tenure (0-1 year)": {
        "insight": "New customers (0-12 months) have highest churn risk (~47%)",
        "actions": [
            "Onboarding program with dedicated success manager for first 90 days",
            "Early warning system: trigger outreach if usage drops in first 3 months",
            "Welcome package with service tutorials and proactive check-ins"
        ]
    },
    "Payment Method (Electronic Check)": {
        "insight": "Electronic check users churn at ~45% vs ~15-20% for auto-pay methods",
        "actions": [
            "Incentivize auto-pay enrollment (monthly discount, bill credit)",
            "Simplify auto-pay setup process",
            "Targeted campaign for electronic check users highlighting benefits"
        ]
    },
    "Internet Service (Fiber Optic)": {
        "insight": "Fiber optic customers show higher churn (~42%) than DSL (~19%)",
        "actions": [
            "Investigate service quality issues for fiber customers",
            "Offer technical support proactively for fiber installations",
            "Bundle fiber with security/backup services to increase switching cost"
        ]
    },
    "Online Security / Tech Support / Device Protection (No)": {
        "insight": "Customers without add-on services churn 2-3x more than those with them",
        "actions": [
            "Bundle security services at discounted rate",
            "Free trial of premium support for at-risk customers",
            "Educate customers on value of protection services"
        ]
    },
    "High Monthly Charges": {
        "insight": "Customers with above-average monthly charges more likely to churn",
        "actions": [
            "Bill review and optimization recommendations",
            "Customized plan recommendations based on usage patterns",
            "Loyalty discounts for long-term high-value customers"
        ]
    }
}

for driver, details in recommendations.items():
    print(f"\n{'='*60}")
    print(f"DRIVER: {driver}")
    print(f"INSIGHT: {details['insight']}")
    print(f"RECOMMENDED ACTIONS:")
    for i, action in enumerate(details['actions'], 1):
        print(f"  {i}. {action}")

## 9. Model Selection & Final Model Persistence

In [ ]:
# Select best model based on business criteria
print("MODEL SELECTION SUMMARY")
print("="*60)

selection_df = pd.DataFrame()
for model_name, results in evaluation_results.items():
    selection_df.loc[model_name, 'Threshold'] = f"{results['threshold']:.3f}"
    selection_df.loc[model_name, 'Accuracy'] = f"{results['accuracy']:.4f}"
    selection_df.loc[model_name, 'Precision'] = f"{results['precision']:.4f}"
    selection_df.loc[model_name, 'Recall'] = f"{results['recall']:.4f}"
    selection_df.loc[model_name, 'F1-Score'] = f"{results['f1']:.4f}"
    selection_df.loc[model_name, 'ROC-AUC'] = f"{results['roc_auc']:.4f}"
    selection_df.loc[model_name, 'Pred_Churners'] = results['predicted_churners']

print(selection_df.to_string())

# Select based on recall at capacity (primary) then F1
best_model_name = max(evaluation_results.keys(), 
                       key=lambda k: (evaluation_results[k]['recall'], evaluation_results[k]['f1']))

print(f"\nSELECTED MODEL: {best_model_name}")
print(f"Reason: Highest recall at capacity threshold with strong F1")

In [ ]:
# Save final model, encoders, scaler, and threshold
final_model = best_models[best_model_name]
final_threshold = evaluation_results[best_model_name]['threshold']

joblib.dump(final_model, "best_model.joblib")
joblib.dump(encoders, "encoders.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(final_threshold, "threshold.joblib")
joblib.dump(feature_names, "feature_names.joblib")

print(f"Saved:")
print(f"  - best_model.joblib ({best_model_name})")
print(f"  - encoders.joblib")
print(f"  - scaler.joblib")
print(f"  - threshold.joblib (value: {final_threshold:.4f})")
print(f"  - feature_names.joblib")

In [ ]:
# Updated prediction function with optimal threshold
def make_prediction(input_data, threshold=None):
    """Make churn prediction with configurable threshold."""
    if threshold is None:
        threshold = joblib.load("threshold.joblib")
    
    input_df = pd.DataFrame([input_data])
    
    # Apply feature engineering (must match training)
    # Tenure bands
    input_df['tenure_band'] = pd.cut(input_df['tenure'], 
                                      bins=[0, 12, 24, 48, 72], 
                                      labels=['0-1yr', '1-2yr', '2-4yr', '4+yr'],
                                      right=True)
    
    # Avg monthly charges
    input_df['avg_monthly_charges'] = input_df['TotalCharges'] / (input_df['tenure'] + 1)
    
    # High value customer
    # Note: In production, use training quantile threshold
    high_value_threshold = df_fe['TotalCharges'].quantile(0.75)
    input_df['high_value_customer'] = (input_df['TotalCharges'] > high_value_threshold).astype(int)
    
    # Service count
    service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    
    def count_active_services(row):
        count = 0
        for col in service_cols:
            val = row[col]
            if val not in ['No', 'No internet service', 'No phone service']:
                count += 1
        return count
    
    input_df['service_count'] = input_df.apply(count_active_services, axis=1)
    
    # Contract encoded
    contract_map = {'Month-to-month': 1, 'One year': 2, 'Two year': 3}
    input_df['contract_encoded'] = input_df['Contract'].map(contract_map)
    
    # Payment risk
    payment_risk = {'Electronic check': 3, 'Mailed check': 2, 'Bank transfer (automatic)': 1, 'Credit card (automatic)': 1}
    input_df['payment_risk_score'] = input_df['PaymentMethod'].map(payment_risk)
    
    # Interactions
    input_df['senior_with_dependents'] = ((input_df['SeniorCitizen'] == 1) & (input_df['Dependents'] == 'Yes')).astype(int)
    input_df['paperless_electronic'] = ((input_df['PaperlessBilling'] == 'Yes') & (input_df['PaymentMethod'] == 'Electronic check')).astype(int)
    
    # Encode categorical
    for col, encoder in encoders.items():
        if col in input_df.columns:
            # Handle unseen categories
            input_df[col] = input_df[col].astype(str)
            # Transform with handling for unseen
            known_classes = set(encoder.classes_)
            input_df[col] = input_df[col].apply(lambda x: x if x in known_classes else encoder.classes_[0])
            input_df[col] = encoder.transform(input_df[col])
    
    # Scale numerical
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_charges']
    input_df[num_cols] = scaler.transform(input_df[num_cols])
    
    # Ensure column order matches training
    input_df = input_df[feature_names]
    
    # Predict
    probability = final_model.predict_proba(input_df)[0, 1]
    prediction = 1 if probability >= threshold else 0
    
    return "Churn" if prediction == 1 else "No Churn", probability

# Test with example
example_input = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 1,
    'PhoneService': 'No',
    'MultipleLines': 'No phone service',
    'InternetService': 'DSL',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'Yes',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'No',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 70.35,
    'TotalCharges': 29.85
}

prediction, prob = make_prediction(example_input)
print(f"Prediction: {prediction}, Probability: {prob:.4f}, Threshold: {final_threshold:.4f}")

## 10. Generate Churn Risk Output for Power BI

In [ ]:
# Generate churn risk scores for entire dataset
print("Generating churn risk scores for all customers...")

# Prepare full dataset with feature engineering
df_full = df_fe.copy()
df_full['Churn'] = df_full['Churn'].replace({1: 'Yes', 0: 'No'})  # Restore labels

# Predict on full dataset
X_full = df_full.drop(columns=['Churn'])

# Encode categorical
for col in cat_cols:
    X_full[col] = encoders[col].transform(X_full[col].astype(str))

# Scale numerical
X_full[num_cols] = scaler.transform(X_full[num_cols])

# Ensure column order
X_full = X_full[feature_names]

# Get probabilities
churn_probabilities = final_model.predict_proba(X_full)[:, 1]
churn_predictions = (churn_probabilities >= final_threshold).astype(int)

# Add to dataframe
df_full['churn_probability'] = churn_probabilities
df_full['churn_risk_score'] = churn_predictions
df_full['risk_category'] = pd.cut(churn_probabilities, 
                                    bins=[0, 0.3, 0.7, 1.0], 
                                    labels=['Low', 'Medium', 'High'])

# Revenue at risk
df_full['revenue_at_risk_monthly'] = df_full['MonthlyCharges'] * df_full['churn_probability']
df_full['revenue_at_risk_annual'] = df_full['revenue_at_risk_monthly'] * 12

# Save for Power BI
output_cols = ['customerID'] if 'customerID' in df_full.columns else []
output_cols += ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
                'PhoneService', 'MultipleLines', 'InternetService', 'Contract',
                'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges',
                'Churn', 'churn_probability', 'churn_risk_score', 'risk_category',
                'revenue_at_risk_monthly', 'revenue_at_risk_annual']

# Filter to existing columns
output_cols = [c for c in output_cols if c in df_full.columns]
df_output = df_full[output_cols].copy()

df_output.to_csv("churn_risk_output.csv", index=False)
print(f"Saved churn_risk_output.csv with {len(df_output)} rows")
print(f"Columns: {list(df_output.columns)}")

# Summary statistics
print(f"\nRisk Distribution:")
print(df_output['risk_category'].value_counts())
print(f"\nTotal Monthly Revenue at Risk: ${df_output['revenue_at_risk_monthly'].sum():,.2f}")
print(f"Total Annual Revenue at Risk: ${df_output['revenue_at_risk_annual'].sum():,.2f}")
print(f"High-Risk Customers: {(df_output['risk_category'] == 'High').sum()}")
print(f"High-Risk Monthly Revenue: ${df_output[df_output['risk_category'] == 'High']['revenue_at_risk_monthly'].sum():,.2f}")

## 11. Power BI Dashboard Specification

In [ ]:
# Print DAX measures specification for Power BI
dax_measures = """
-- POWER BI DAX MEASURES SPECIFICATION
-- Copy these into Power BI Desktop > Modeling > New Measure

-- 1. CHURN RATE
Churn Rate = 
DIVIDE(
    CALCULATE(COUNTROWS('ChurnData'), 'ChurnData'[Churn] = "Yes"),
    COUNTROWS('ChurnData')
)

-- 2. TOTAL CUSTOMERS
Total Customers = COUNTROWS('ChurnData')

-- 3. CHURNED CUSTOMERS
Churned Customers = CALCULATE(COUNTROWS('ChurnData'), 'ChurnData'[Churn] = "Yes")

-- 4. MONTHLY REVENUE AT RISK
Monthly Revenue at Risk = SUMX('ChurnData', 'ChurnData'[MonthlyCharges] * 'ChurnData'[churn_probability])

-- 5. ANNUAL REVENUE AT RISK
Annual Revenue at Risk = [Monthly Revenue at Risk] * 12

-- 6. HIGH-RISK CUSTOMERS
High Risk Customers = CALCULATE(COUNTROWS('ChurnData'), 'ChurnData'[risk_category] = "High")

-- 7. HIGH-RISK REVENUE
High Risk Revenue = CALCULATE(SUM('ChurnData'[revenue_at_risk_monthly]), 'ChurnData'[risk_category] = "High")

-- 8. CHURN RATE BY CONTRACT
Churn Rate by Contract = 
DIVIDE(
    CALCULATE(COUNTROWS('ChurnData'), 'ChurnData'[Churn] = "Yes"),
    CALCULATE(COUNTROWS('ChurnData'), ALLEXCEPT('ChurnData', 'ChurnData'[Contract]))
)

-- 9. CHURN RATE BY TENURE BAND
Churn Rate by Tenure Band = 
DIVIDE(
    CALCULATE(COUNTROWS('ChurnData'), 'ChurnData'[Churn] = "Yes"),
    CALCULATE(COUNTROWS('ChurnData'), ALLEXCEPT('ChurnData', 'ChurnData'[tenure_band]))
)

-- 10. CHURN RATE BY PAYMENT METHOD
Churn Rate by Payment Method = 
DIVIDE(
    CALCULATE(COUNTROWS('ChurnData'), 'ChurnData'[Churn] = "Yes"),
    CALCULATE(COUNTROWS('ChurnData'), ALLEXCEPT('ChurnData', 'ChurnData'[PaymentMethod]))
)

-- 11. RETENTION ROI (if intervention cost known)
-- Assumption: $50 intervention cost per customer, 30% save rate
Retention ROI = 
VAR InterventionCost = 50
VAR SaveRate = 0.30
VAR CustomersTargeted = [High Risk Customers]
VAR MonthlyRevenueSaved = [High Risk Revenue] * SaveRate
VAR AnnualRevenueSaved = MonthlyRevenueSaved * 12
VAR TotalCost = CustomersTargeted * InterventionCost
RETURN
DIVIDE(AnnualRevenueSaved - TotalCost, TotalCost)

-- 12. AVERAGE CHURN PROBABILITY
Avg Churn Probability = AVERAGE('ChurnData'[churn_probability])

-- 13. CUSTOMER LIFETIME VALUE (simplified)
Avg Customer Lifetime Value = 
DIVIDE(
    SUM('ChurnData'[TotalCharges]),
    COUNTROWS('ChurnData')
)

-- 14. TENURE DISTRIBUTION
Avg Tenure = AVERAGE('ChurnData'[tenure])

-- 15. MONTHLY CHARGES DISTRIBUTION
Avg Monthly Charges = AVERAGE('ChurnData'[MonthlyCharges])
"""

print(dax_measures)

## 12. Summary & Next Steps

In [ ]:
# Final summary
print("="*80)
print("PROJECT COMPLETION SUMMARY")
print("="*80)

print(f"\nDATA INGESTION: Loaded Telco dataset (7,043 customers, 20 features)")
print(f"EDA & DATA QUALITY: Missing values handled, distributions analyzed, outliers detected via IQR")
print(f"FEATURE ENGINEERING: 8 new features created (tenure bands, service count, customer value, risk scores, interactions)")
print(f"PREDICTIVE MODELING: 5 classifiers trained with stratified 5-fold CV + GridSearchCV")
print(f"   - Logistic Regression")
print(f"   - Decision Tree")
print(f"   - Random Forest")
print(f"   - Neural Network (MLP)")
print(f"   - XGBoost")
print(f"CLASS IMBALANCE: Addressed with SMOTE")
print(f"THRESHOLD TUNING: Optimized for retention team capacity (20%)")
print(f"MODEL EVALUATION: Precision, Recall, F1, ROC-AUC, Confusion Matrix for all models")
print(f"FEATURE IMPORTANCE: Permutation importance + SHAP (if available)")
print(f"CHURN DRIVERS: Top 10 identified with business insights")
print(f"RETENTION RECOMMENDATIONS: 6 action areas with specific tactics")
print(f"MODEL PERSISTENCE: Best model, encoders, scaler, threshold saved")
print(f"CHURN RISK OUTPUT: churn_risk_output.csv generated for Power BI")
print(f"POWER BI SPEC: DAX measures documented for dashboard creation")

print(f"\nSELECTED MODEL: {best_model_name}")
print(f"   Threshold: {final_threshold:.4f}")
print(f"   Recall at capacity: {evaluation_results[best_model_name]['recall']:.4f}")
print(f"   Precision at capacity: {evaluation_results[best_model_name]['precision']:.4f}")
print(f"   F1-Score: {evaluation_results[best_model_name]['f1']:.4f}")
print(f"   ROC-AUC: {evaluation_results[best_model_name]['roc_auc']:.4f}")

print(f"\nKEY FINDINGS:")
print(f"  - Month-to-month contracts: 42% churn vs 3% for 2-year")
print(f"  - New customers (0-1yr): 47% churn rate")
print(f"  - Electronic check payments: 45% churn")
print(f"  - No online security/tech support: 2-3x higher churn")
print(f"  - Fiber optic customers: Higher churn than DSL")

print(f"\nDELIVERABLES CREATED:")
print(f"  - best_model.joblib (trained {best_model_name})")
print(f"  - encoders.joblib (LabelEncoders for categorical features)")
print(f"  - scaler.joblib (StandardScaler for numerical features)")
print(f"  - threshold.joblib (optimal decision threshold)")
print(f"  - feature_names.joblib (feature column order)")
print(f"  - churn_risk_output.csv (risk scores for all customers)")
print(f"  - Customer_Churn_Prediction_Enhanced.ipynb (this notebook)")

print(f"\nNEXT STEPS FOR POWER BI DASHBOARD:")
print(f"  1. Open Power BI Desktop")
print(f"  2. Get Data -> CSV -> churn_risk_output.csv")
print(f"  3. Create DAX measures from specification above")
print(f"  4. Build visuals:")
print(f"     - KPI cards: Churn Rate, Total Customers, Revenue at Risk")
print(f"     - Bar charts: Churn by Contract, Tenure Band, Payment Method")
print(f"     - Scatter: Monthly Charges vs Churn Probability")
print(f"     - Table: Top 50 High-Risk Customers with Revenue at Risk")
print(f"     - Gauge: Retention ROI")
print(f"  5. Add slicers: Contract, Tenure Band, Payment Method, Risk Category")
print(f"  6. Publish to Power BI Service for sharing")